<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/feature%2FSCRUM-13/notebooks/ClimatePipeline/02_ClimateDataAudit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClimateDataAudit

Auditoría genérica y de solo lectura para los Parquet climáticos del proyecto RAIZ.

Este notebook no limpia ni transforma los datos crudos. Produce un diagnóstico reproducible para definir las reglas de limpieza y preparar la construcción posterior de registros diarios.

## Alcance del diagnóstico

- Inventario exacto mediante metadatos Parquet.
- Validación de esquema y tipos entre archivos.
- Muestras estratificadas por departamento, año y mes.
- Bloques contiguos de archivos para estudiar frecuencia sin mezclar saltos artificiales.
- Nulos, conversiones fallidas, unidades, valores y coordenadas.
- Duplicados exactos, duplicados de clave y conflictos.
- Cobertura y frecuencia por estación y sensor.
- Conteo completo opcional por estación y sensor.
- Hallazgos clasificados por severidad.

La auditoría no imputa datos ni decide una agregación diaria. Esas reglas dependen de la variable y pertenecen al siguiente componente del pipeline.


## 1. Configuración

Configure una variable y su fuente Socrata. Los filtros de departamento, año y mes determinan qué particiones se auditan.

La bandera `EJECUTAR_AUDITORIA` permanece desactivada para que Run all sea seguro. El conteo completo por estación es opcional porque debe leer todos los archivos seleccionados.


In [20]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

try:
    from google.colab import drive

    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Deben coincidir con la carpeta creada por 01_ClimateDataDownloader.
DATASET_ID = 'uext-mhny'
VARIABLE_NOMBRE = 'humedad'

AUDITORIA_DEPARTAMENTOS = ['CUNDINAMARCA']
AUDITORIA_ANIOS = [2025]
AUDITORIA_MESES = None  # None usa todos los meses disponibles.

# Muestra: dos bloques aleatorios de dos archivos contiguos por partición.
BLOQUES_POR_PARTICION = 2
ARCHIVOS_POR_BLOQUE = 2
MAX_FILAS_MUESTRA = 250_000
SEMILLA_MUESTRA = 2026

# None solo describe la distribución. Ejemplo posterior: (300, 1100).
RANGO_PLAUSIBLE = None

# Lee todos los Parquet, pero únicamente estación, sensor y fecha.
EJECUTAR_CONTEO_ESTACIONES_COMPLETO = True

# Inventario rápido de todas las variables; solo cuenta nombres de archivos.
EJECUTAR_INVENTARIO_GENERAL = False

# Guardar resultados es opcional; nunca modifica clima_crudo.
GUARDAR_RESULTADOS = False
ETIQUETA_SALIDA = 'diagnostico'

# Banderita de seguridad para Run all.
EJECUTAR_AUDITORIA = True

PROCESSED_ROOT = (
    Path('/content/drive/MyDrive/eco2026_processed')
    if IN_COLAB
    else Path.cwd() / 'eco2026_processed'
)

COLUMNAS_ESPERADAS = [
    'codigoestacion',
    'codigosensor',
    'dataset_id',
    'departamento',
    'descripcionsensor',
    'fechaobservacion',
    'latitud',
    'longitud',
    'municipio',
    'nombreestacion',
    'unidadmedida',
    'valorobservado',
    'zonahidrografica',
]

print({
    'dataset_id': DATASET_ID,
    'variable': VARIABLE_NOMBRE,
    'departamentos': AUDITORIA_DEPARTAMENTOS,
    'anios': AUDITORIA_ANIOS,
    'meses': AUDITORIA_MESES,
    'bloques_por_particion': BLOQUES_POR_PARTICION,
    'archivos_por_bloque': ARCHIVOS_POR_BLOQUE,
    'conteo_estaciones_completo': EJECUTAR_CONTEO_ESTACIONES_COMPLETO,
    'inventario_general': EJECUTAR_INVENTARIO_GENERAL,
    'guardar_resultados': GUARDAR_RESULTADOS,
    'ejecutar_auditoria': EJECUTAR_AUDITORIA,
    'processed_root': str(PROCESSED_ROOT),
})


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'dataset_id': 'uext-mhny', 'variable': 'humedad', 'departamentos': ['CUNDINAMARCA'], 'anios': [2025], 'meses': None, 'bloques_por_particion': 2, 'archivos_por_bloque': 2, 'conteo_estaciones_completo': True, 'inventario_general': False, 'guardar_resultados': False, 'ejecutar_auditoria': True, 'processed_root': '/content/drive/MyDrive/eco2026_processed'}


## 2. Inventario general aproximado

Este inventario recorre todas las variables dentro de `clima_crudo`, pero no abre los Parquet. Cuenta archivos `part-*` y usa el máximo de 1.000 filas por archivo para producir una estimación superior rápida.

Los años y departamentos se deducen de la estructura de carpetas. El total real puede ser menor porque el último archivo de cada partición suele contener menos de 1.000 filas.


In [21]:
def etiquetas_desde_ruta(archivo, raiz):
    etiquetas = {}
    for parte in archivo.relative_to(raiz).parts:
        if '=' in parte:
            clave, valor = parte.split('=', 1)
            etiquetas[clave] = valor
    return etiquetas


def construir_inventario_general(raiz_clima, filas_maximas_por_archivo=1000):
    if not raiz_clima.exists():
        return pd.DataFrame()

    acumulado = {}
    for archivo in raiz_clima.rglob('part-*.parquet'):
        etiquetas = etiquetas_desde_ruta(archivo, raiz_clima)
        variable = etiquetas.get('variable')
        fuente = etiquetas.get('fuente')
        if not variable or not fuente:
            continue

        clave = (variable, fuente)
        if clave not in acumulado:
            acumulado[clave] = {
                'lotes_archivos': 0,
                'anios': set(),
                'departamentos': set(),
            }

        actual = acumulado[clave]
        actual['lotes_archivos'] += 1
        if str(etiquetas.get('anio', '')).isdigit():
            actual['anios'].add(int(etiquetas['anio']))
        if etiquetas.get('departamento'):
            actual['departamentos'].add(etiquetas['departamento'])

    filas = []
    for (variable, fuente), valores in sorted(acumulado.items()):
        anios = sorted(valores['anios'])
        estimado = valores['lotes_archivos'] * int(filas_maximas_por_archivo)
        filas.append({
            'variable': variable,
            'fuente': fuente,
            'lotes_archivos': valores['lotes_archivos'],
            'registros_estimados': f'~{estimado:,}',
            'registros_estimados_max': estimado,
            'anio_inicio': anios[0] if anios else None,
            'anio_fin': anios[-1] if anios else None,
            'departamentos': ', '.join(sorted(valores['departamentos'])),
        })

    return pd.DataFrame(filas)


inventario_general_descargas = pd.DataFrame()
if not EJECUTAR_INVENTARIO_GENERAL:
    print(
        'Inventario general desactivado. Cambie '
        'EJECUTAR_INVENTARIO_GENERAL a True para generar la tabla.'
    )
else:
    raiz_clima = PROCESSED_ROOT / 'clima_crudo'
    inventario_general_descargas = construir_inventario_general(raiz_clima)
    if inventario_general_descargas.empty:
        print(f'No se encontraron archivos part-*.parquet en {raiz_clima}.')
    else:
        display(inventario_general_descargas)


Inventario general desactivado. Cambie EJECUTAR_INVENTARIO_GENERAL a True para generar la tabla.


## 3. Descubrimiento e inventario Parquet

El inventario recorre los archivos seleccionados y consulta sus metadatos sin cargar las filas. Los conteos de archivos, filas y bytes de esta sección son exactos.

También se comprueba la secuencia de archivos part-xxxxx.parquet dentro de cada partición.


In [22]:
import json
import random
import re
import unicodedata
from collections import defaultdict

PART_PATTERN = re.compile(r'^part-(\d{5})\.parquet$')
DEPARTAMENTOS_PERMITIDOS = {'BOYACÁ', 'CUNDINAMARCA'}


def slugificar(valor):
    texto = unicodedata.normalize('NFKD', str(valor))
    texto = texto.encode('ascii', errors='ignore').decode('ascii').lower()
    texto = re.sub(r'[^a-z0-9]+', '_', texto).strip('_')
    if not texto:
        raise ValueError(f'No se pudo construir una etiqueta para {valor!r}.')
    return texto


def normalizar_lista_enteros(valores, minimo, maximo, nombre):
    if valores is None:
        return None
    normalizados = sorted({int(valor) for valor in valores})
    fuera_rango = [valor for valor in normalizados if valor < minimo or valor > maximo]
    if fuera_rango:
        raise ValueError(f'{nombre} fuera de rango: {fuera_rango}.')
    return normalizados


def validar_configuracion():
    departamentos = sorted({
        str(departamento).strip().upper()
        for departamento in AUDITORIA_DEPARTAMENTOS
    })
    if not departamentos:
        raise ValueError('AUDITORIA_DEPARTAMENTOS no puede estar vacío.')

    no_permitidos = set(departamentos) - DEPARTAMENTOS_PERMITIDOS
    if no_permitidos:
        raise ValueError(f'Departamentos fuera del alcance: {sorted(no_permitidos)}.')

    anios = normalizar_lista_enteros(AUDITORIA_ANIOS, 1900, 2100, 'Años')
    meses = normalizar_lista_enteros(AUDITORIA_MESES, 1, 12, 'Meses')

    if int(BLOQUES_POR_PARTICION) <= 0:
        raise ValueError('BLOQUES_POR_PARTICION debe ser positivo.')
    if int(ARCHIVOS_POR_BLOQUE) <= 0:
        raise ValueError('ARCHIVOS_POR_BLOQUE debe ser positivo.')
    if int(MAX_FILAS_MUESTRA) <= 0:
        raise ValueError('MAX_FILAS_MUESTRA debe ser positivo.')
    if RANGO_PLAUSIBLE is not None:
        if len(RANGO_PLAUSIBLE) != 2 or RANGO_PLAUSIBLE[0] >= RANGO_PLAUSIBLE[1]:
            raise ValueError('RANGO_PLAUSIBLE debe ser None o una pareja (mínimo, máximo).')

    return departamentos, anios, meses


def raiz_dataset():
    return (
        PROCESSED_ROOT
        / 'clima_crudo'
        / f'variable={slugificar(VARIABLE_NOMBRE)}'
        / f'fuente={str(DATASET_ID).lower()}'
    )


def particiones_desde_ruta(archivo, raiz):
    valores = {}
    for parte in archivo.relative_to(raiz).parts:
        if '=' in parte:
            clave, valor = parte.split('=', 1)
            valores[clave] = valor

    try:
        anio = int(valores.get('anio'))
        mes = int(valores.get('mes'))
    except (TypeError, ValueError):
        anio = None
        mes = None

    return {
        'departamento': valores.get('departamento'),
        'anio': anio,
        'mes': mes,
    }


def descubrir_archivos(raiz, departamentos, anios=None, meses=None):
    if not raiz.exists():
        return []

    archivos = []
    for archivo in raiz.rglob('*.parquet'):
        particion = particiones_desde_ruta(archivo, raiz)
        if particion['departamento'] not in departamentos:
            continue
        if anios is not None and particion['anio'] not in anios:
            continue
        if meses is not None and particion['mes'] not in meses:
            continue
        archivos.append(archivo)

    return sorted(
        archivos,
        key=lambda archivo: (
            particiones_desde_ruta(archivo, raiz)['departamento'] or '',
            particiones_desde_ruta(archivo, raiz)['anio'] or -1,
            particiones_desde_ruta(archivo, raiz)['mes'] or -1,
            archivo.name,
        ),
    )


def construir_inventario(archivos, raiz):
    try:
        import pyarrow.parquet as pq
    except ImportError as exc:
        raise ImportError('Se necesita pyarrow para auditar Parquet.') from exc

    filas = []
    for archivo in archivos:
        particion = particiones_desde_ruta(archivo, raiz)
        coincidencia = PART_PATTERN.fullmatch(archivo.name)

        try:
            parquet = pq.ParquetFile(archivo)
            esquema = {
                campo.name: str(campo.type)
                for campo in parquet.schema_arrow
            }
            filas.append({
                **particion,
                'parte': int(coincidencia.group(1)) if coincidencia else None,
                'archivo': str(archivo),
                'filas': parquet.metadata.num_rows,
                'tamano_bytes': archivo.stat().st_size,
                'numero_columnas': len(esquema),
                'columnas': tuple(esquema),
                'esquema': esquema,
                'firma_esquema': json.dumps(esquema, sort_keys=True, ensure_ascii=False),
                'error_lectura': None,
            })
        except Exception as exc:
            filas.append({
                **particion,
                'parte': int(coincidencia.group(1)) if coincidencia else None,
                'archivo': str(archivo),
                'filas': None,
                'tamano_bytes': archivo.stat().st_size if archivo.exists() else None,
                'numero_columnas': None,
                'columnas': tuple(),
                'esquema': {},
                'firma_esquema': None,
                'error_lectura': f'{type(exc).__name__}: {exc}',
            })

    return pd.DataFrame(filas)


def resumir_particiones(inventario):
    if inventario.empty:
        return pd.DataFrame()

    registros = []
    claves = ['departamento', 'anio', 'mes']
    for clave, grupo in inventario.groupby(claves, dropna=False, sort=True):
        partes = sorted(grupo['parte'].dropna().astype(int).tolist())
        faltantes = []
        if partes:
            esperadas = set(range(partes[-1] + 1))
            faltantes = sorted(esperadas - set(partes))

        registros.append({
            **dict(zip(claves, clave)),
            'archivos': len(grupo),
            'filas': int(grupo['filas'].fillna(0).sum()),
            'tamano_mb': round(grupo['tamano_bytes'].fillna(0).sum() / (1024 ** 2), 2),
            'primera_parte': partes[0] if partes else None,
            'ultima_parte': partes[-1] if partes else None,
            'partes_faltantes': faltantes,
            'errores_lectura': int(grupo['error_lectura'].notna().sum()),
        })

    return pd.DataFrame(registros)


def resumir_esquemas(inventario):
    if inventario.empty:
        return pd.DataFrame(), pd.DataFrame()

    archivos_validos = inventario[inventario['error_lectura'].isna()]
    presencia = []
    tipos = []

    for columna in sorted(set().union(*archivos_validos['columnas'].tolist())):
        presencia.append({
            'columna': columna,
            'archivos_presente': int(archivos_validos['columnas'].apply(
                lambda columnas: columna in columnas
            ).sum()),
            'archivos_totales': len(archivos_validos),
            'esperada': columna in COLUMNAS_ESPERADAS,
        })

        conteo_tipos = defaultdict(int)
        for esquema in archivos_validos['esquema']:
            if columna in esquema:
                conteo_tipos[esquema[columna]] += 1
        for tipo, cantidad in sorted(conteo_tipos.items()):
            tipos.append({
                'columna': columna,
                'tipo_parquet': tipo,
                'archivos': cantidad,
            })

    return pd.DataFrame(presencia), pd.DataFrame(tipos)


## 4. Selección de muestras estratificadas

Cada combinación departamento + año + mes es un estrato. Dentro de cada estrato se seleccionan bloques aleatorios de archivos contiguos.

La continuidad dentro de cada bloque permite estimar intervalos temporales sin interpretar como frecuencia el salto existente entre dos muestras alejadas. La semilla hace reproducible la selección.


In [23]:
def seleccionar_archivos_muestra(inventario):
    validos = inventario[
        inventario['error_lectura'].isna()
        & inventario['parte'].notna()
    ].copy()
    if validos.empty:
        return pd.DataFrame()

    rng = random.Random(int(SEMILLA_MUESTRA))
    seleccion = []
    claves = ['departamento', 'anio', 'mes']

    for clave, grupo in validos.groupby(claves, sort=True):
        grupo = grupo.sort_values('parte').reset_index(drop=True)
        cantidad = len(grupo)
        tamano_bloque = min(int(ARCHIVOS_POR_BLOQUE), cantidad)
        candidatos = list(range(0, cantidad - tamano_bloque + 1))
        rng.shuffle(candidatos)

        ocupados = set()
        inicios = []
        for inicio in candidatos:
            indices = set(range(inicio, inicio + tamano_bloque))
            if indices & ocupados:
                continue
            inicios.append(inicio)
            ocupados.update(indices)
            if len(inicios) >= int(BLOQUES_POR_PARTICION):
                break

        if not inicios:
            inicios = [0]

        for numero_bloque, inicio in enumerate(sorted(inicios), start=1):
            bloque = grupo.iloc[inicio:inicio + tamano_bloque].copy()
            bloque['bloque_muestra'] = (
                f'{clave[0]}-{clave[1]}-{int(clave[2]):02d}-B{numero_bloque}'
            )
            seleccion.append(bloque)

    resultado = pd.concat(seleccion, ignore_index=True)
    resultado = resultado.drop_duplicates(subset='archivo')
    return resultado.sort_values(
        ['departamento', 'anio', 'mes', 'parte']
    ).reset_index(drop=True)


def cargar_muestra(seleccion):
    if seleccion.empty:
        return pd.DataFrame()

    filas_estimadas = int(seleccion['filas'].fillna(0).sum())
    if filas_estimadas > int(MAX_FILAS_MUESTRA):
        raise RuntimeError(
            f'La muestra seleccionada tendría {filas_estimadas:,} filas, '
            f'más que MAX_FILAS_MUESTRA={MAX_FILAS_MUESTRA:,}. '
            'Reduzca bloques, archivos por bloque o el alcance temporal.'
        )

    bloques = []
    for fila in seleccion.itertuples(index=False):
        columnas_archivo = list(fila.columnas)
        columnas_lectura = [
            columna for columna in COLUMNAS_ESPERADAS
            if columna in columnas_archivo
        ]
        bloque = pd.read_parquet(fila.archivo, columns=columnas_lectura)

        for columna in COLUMNAS_ESPERADAS:
            if columna not in bloque.columns:
                bloque[columna] = pd.NA

        bloque = bloque[COLUMNAS_ESPERADAS]
        bloque['_archivo'] = fila.archivo
        bloque['_bloque_muestra'] = fila.bloque_muestra
        bloque['_departamento_particion'] = fila.departamento
        bloque['_anio_particion'] = fila.anio
        bloque['_mes_particion'] = fila.mes
        bloques.append(bloque)

    return pd.concat(bloques, ignore_index=True)


def preparar_muestra(muestra):
    preparada = muestra.copy()

    fecha_original_valida = preparada['fechaobservacion'].notna()
    valor_original_valido = preparada['valorobservado'].notna()
    latitud_original_valida = preparada['latitud'].notna()
    longitud_original_valida = preparada['longitud'].notna()

    preparada['fechaobservacion_dt'] = pd.to_datetime(
        preparada['fechaobservacion'],
        errors='coerce',
    )
    preparada['valorobservado_num'] = pd.to_numeric(
        preparada['valorobservado'],
        errors='coerce',
    )
    preparada['latitud_num'] = pd.to_numeric(preparada['latitud'], errors='coerce')
    preparada['longitud_num'] = pd.to_numeric(preparada['longitud'], errors='coerce')

    preparada['_error_fecha'] = (
        fecha_original_valida & preparada['fechaobservacion_dt'].isna()
    )
    preparada['_error_valor'] = (
        valor_original_valido & preparada['valorobservado_num'].isna()
    )
    preparada['_error_latitud'] = (
        latitud_original_valida & preparada['latitud_num'].isna()
    )
    preparada['_error_longitud'] = (
        longitud_original_valida & preparada['longitud_num'].isna()
    )

    return preparada


## 5. Calidad de campos, duplicados y conflictos

La clave técnica provisional es:

codigoestacion + codigosensor + fechaobservacion

Se distinguen tres síntomas:

- Duplicado exacto: las 13 columnas coinciden.
- Duplicado de clave: la clave técnica aparece más de una vez.
- Conflicto: una clave repetida contiene valores observados diferentes.

La auditoría solo reporta estos casos; no elimina filas.


In [24]:
CLAVE_OBSERVACION = [
    'codigoestacion',
    'codigosensor',
    'fechaobservacion_dt',
]


def auditar_nulos(muestra):
    total = len(muestra)
    return pd.DataFrame([
        {
            'columna': columna,
            'nulos': int(muestra[columna].isna().sum()),
            'porcentaje_nulos': round(
                muestra[columna].isna().mean() * 100,
                2,
            ) if total else 0.0,
        }
        for columna in COLUMNAS_ESPERADAS
    ])


def auditar_conversiones(muestra):
    return pd.DataFrame([
        {'campo': 'fechaobservacion', 'conversiones_fallidas': int(muestra['_error_fecha'].sum())},
        {'campo': 'valorobservado', 'conversiones_fallidas': int(muestra['_error_valor'].sum())},
        {'campo': 'latitud', 'conversiones_fallidas': int(muestra['_error_latitud'].sum())},
        {'campo': 'longitud', 'conversiones_fallidas': int(muestra['_error_longitud'].sum())},
    ])


def auditar_coordenadas_y_particion(muestra):
    latitud_invalida = muestra['latitud_num'].notna() & ~muestra['latitud_num'].between(-90, 90)
    longitud_invalida = muestra['longitud_num'].notna() & ~muestra['longitud_num'].between(-180, 180)
    departamento_dato = muestra['departamento'].fillna('<NULO>').astype(str).str.strip().str.upper()
    departamento_particion = (
        muestra['_departamento_particion'].fillna('<NULO>').astype(str).str.strip().str.upper()
    )
    departamento_inconsistente = departamento_dato != departamento_particion

    return pd.DataFrame([
        {'metrica': 'latitudes_fuera_rango', 'registros_muestra': int(latitud_invalida.sum())},
        {'metrica': 'longitudes_fuera_rango', 'registros_muestra': int(longitud_invalida.sum())},
        {
            'metrica': 'departamento_distinto_particion',
            'registros_muestra': int(departamento_inconsistente.sum()),
        },
    ])


def auditar_unidades(muestra):
    return (
        muestra['unidadmedida']
        .fillna('<NULO>')
        .astype(str)
        .value_counts(dropna=False)
        .rename_axis('unidadmedida')
        .reset_index(name='registros_muestra')
    )


def auditar_valores(muestra):
    valores = muestra['valorobservado_num'].dropna()
    if valores.empty:
        return pd.DataFrame(), pd.DataFrame()

    resumen = pd.DataFrame([{
        'registros_validos': len(valores),
        'minimo': valores.min(),
        'p01': valores.quantile(0.01),
        'p05': valores.quantile(0.05),
        'mediana': valores.median(),
        'media': valores.mean(),
        'p95': valores.quantile(0.95),
        'p99': valores.quantile(0.99),
        'maximo': valores.max(),
    }])

    sospechosos = pd.DataFrame()
    if RANGO_PLAUSIBLE is not None:
        minimo, maximo = RANGO_PLAUSIBLE
        mascara = (muestra['valorobservado_num'] < minimo) | (
            muestra['valorobservado_num'] > maximo
        )
        sospechosos = muestra.loc[
            mascara,
            COLUMNAS_ESPERADAS + ['valorobservado_num', '_archivo'],
        ].copy()

    return resumen, sospechosos


def auditar_duplicados(muestra):
    columnas_exactas = [
        columna for columna in COLUMNAS_ESPERADAS
        if columna in muestra.columns
    ]
    mascara_exactos = muestra.duplicated(
        subset=columnas_exactas,
        keep=False,
    )
    ejemplos_exactos = (
        muestra.loc[mascara_exactos, columnas_exactas + ['_archivo']]
        .sort_values(columnas_exactas[:3])
        .head(50)
    )

    claves_validas = muestra.dropna(subset=CLAVE_OBSERVACION)
    grupos = (
        claves_validas
        .groupby(CLAVE_OBSERVACION, dropna=False)
        .agg(
            registros=('valorobservado_num', 'size'),
            valores_distintos=('valorobservado_num', lambda serie: serie.nunique(dropna=False)),
            valor_min=('valorobservado_num', 'min'),
            valor_max=('valorobservado_num', 'max'),
        )
        .reset_index()
    )

    claves_duplicadas = grupos[grupos['registros'] > 1].copy()
    conflictos = claves_duplicadas[
        claves_duplicadas['valores_distintos'] > 1
    ].copy()

    resumen = pd.DataFrame([{
        'filas_duplicadas_exactas': int(mascara_exactos.sum()),
        'grupos_duplicados_exactos': int(
            muestra.loc[mascara_exactos, columnas_exactas]
            .drop_duplicates()
            .shape[0]
        ),
        'claves_duplicadas': len(claves_duplicadas),
        'claves_con_valores_conflictivos': len(conflictos),
    }])

    return resumen, ejemplos_exactos, claves_duplicadas.head(50), conflictos.head(50)


## 6. Estaciones, sensores, frecuencia y cobertura

Los conteos y frecuencias derivados de la muestra se etiquetan como tales. Los intervalos se calculan dentro de cada bloque contiguo para no introducir saltos artificiales entre muestras.

El conteo completo opcional escanea únicamente codigoestacion, codigosensor y fechaobservacion. Ofrece registros y rango temporal exactos para las particiones seleccionadas, pero puede tardar con variables voluminosas.


In [25]:
def resumir_estaciones_muestra(muestra):
    validos = muestra.dropna(
        subset=['codigoestacion', 'codigosensor', 'fechaobservacion_dt']
    ).copy()
    if validos.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    claves_sensor = ['codigoestacion', 'codigosensor']

    conteos = (
        validos
        .groupby(claves_sensor, dropna=False)
        .agg(
            registros_muestra=('fechaobservacion_dt', 'size'),
            fecha_min_muestra=('fechaobservacion_dt', 'min'),
            fecha_max_muestra=('fechaobservacion_dt', 'max'),
            dias_observados_muestra=('fechaobservacion_dt', lambda serie: serie.dt.date.nunique()),
            unidades_muestra=('unidadmedida', lambda serie: serie.nunique(dropna=False)),
            municipios_muestra=('municipio', lambda serie: serie.nunique(dropna=False)),
        )
        .reset_index()
    )
    conteos['alcance'] = 'muestra'

    ordenados = validos.sort_values(
        ['_bloque_muestra', 'codigoestacion', 'codigosensor', 'fechaobservacion_dt']
    ).copy()
    ordenados['delta_segundos'] = (
        ordenados
        .groupby(
            ['_bloque_muestra', 'codigoestacion', 'codigosensor'],
            dropna=False,
        )['fechaobservacion_dt']
        .diff()
        .dt.total_seconds()
    )

    positivos = ordenados[ordenados['delta_segundos'] > 0].copy()

    def moda_segundos(serie):
        modas = serie.mode()
        return modas.iloc[0] if not modas.empty else pd.NA

    if positivos.empty:
        frecuencia = pd.DataFrame()
    else:
        frecuencia = (
            positivos
            .groupby(claves_sensor, dropna=False)
            .agg(
                intervalos_muestra=('delta_segundos', 'size'),
                intervalo_moda_segundos=('delta_segundos', moda_segundos),
                intervalo_mediano_segundos=('delta_segundos', 'median'),
                intervalo_p10_segundos=('delta_segundos', lambda serie: serie.quantile(0.10)),
                intervalo_p90_segundos=('delta_segundos', lambda serie: serie.quantile(0.90)),
                intervalo_min_segundos=('delta_segundos', 'min'),
                intervalo_max_segundos=('delta_segundos', 'max'),
            )
            .reset_index()
        )
        frecuencia['alcance'] = 'muestra_bloques_contiguos'

    por_dia = validos.copy()
    por_dia['fecha_dia'] = por_dia['fechaobservacion_dt'].dt.floor('D')
    por_dia = (
        por_dia
        .groupby(claves_sensor + ['fecha_dia'], dropna=False)
        .size()
        .reset_index(name='observaciones_dia')
    )
    densidad_diaria = (
        por_dia
        .groupby(claves_sensor, dropna=False)
        .agg(
            dias_en_muestra=('fecha_dia', 'size'),
            observaciones_dia_min=('observaciones_dia', 'min'),
            observaciones_dia_mediana=('observaciones_dia', 'median'),
            observaciones_dia_max=('observaciones_dia', 'max'),
        )
        .reset_index()
    )
    densidad_diaria['alcance'] = 'muestra'

    return conteos, frecuencia, densidad_diaria


def auditar_geografia(muestra):
    base = muestra.dropna(subset=['codigoestacion']).copy()
    if base.empty:
        return pd.DataFrame()

    base['_coordenada'] = list(zip(
        base['latitud_num'].round(5),
        base['longitud_num'].round(5),
    ))

    return (
        base
        .groupby('codigoestacion', dropna=False)
        .agg(
            nombres_estacion=('nombreestacion', lambda serie: serie.nunique(dropna=False)),
            departamentos=('departamento', lambda serie: serie.nunique(dropna=False)),
            municipios=('municipio', lambda serie: serie.nunique(dropna=False)),
            coordenadas_redondeadas=('_coordenada', lambda serie: serie.nunique(dropna=False)),
            latitud_min=('latitud_num', 'min'),
            latitud_max=('latitud_num', 'max'),
            longitud_min=('longitud_num', 'min'),
            longitud_max=('longitud_num', 'max'),
        )
        .reset_index()
    )


def contar_estaciones_completo(archivos):
    acumulado = {}

    for numero, archivo in enumerate(archivos, start=1):
        try:
            import pyarrow.parquet as pq

            columnas = set(pq.ParquetFile(archivo).schema_arrow.names)
            requeridas = {'codigoestacion', 'codigosensor', 'fechaobservacion'}
            if not requeridas.issubset(columnas):
                continue

            bloque = pd.read_parquet(
                archivo,
                columns=['codigoestacion', 'codigosensor', 'fechaobservacion'],
            )
            bloque['fechaobservacion'] = pd.to_datetime(
                bloque['fechaobservacion'],
                errors='coerce',
            )
            for columna in ['codigoestacion', 'codigosensor']:
                bloque[columna] = bloque[columna].astype('string').fillna('<NULO>')

            parcial = (
                bloque
                .groupby(['codigoestacion', 'codigosensor'], dropna=False)
                .agg(
                    registros=('fechaobservacion', 'size'),
                    fecha_min=('fechaobservacion', 'min'),
                    fecha_max=('fechaobservacion', 'max'),
                )
                .reset_index()
            )

            for fila in parcial.itertuples(index=False):
                clave = (fila.codigoestacion, fila.codigosensor)
                if clave not in acumulado:
                    acumulado[clave] = {
                        'registros': 0,
                        'fecha_min': fila.fecha_min,
                        'fecha_max': fila.fecha_max,
                    }

                actual = acumulado[clave]
                actual['registros'] += int(fila.registros)
                if pd.notna(fila.fecha_min):
                    if pd.isna(actual['fecha_min']) or fila.fecha_min < actual['fecha_min']:
                        actual['fecha_min'] = fila.fecha_min
                if pd.notna(fila.fecha_max):
                    if pd.isna(actual['fecha_max']) or fila.fecha_max > actual['fecha_max']:
                        actual['fecha_max'] = fila.fecha_max

        except Exception as exc:
            print(f'ADVERTENCIA al contar {archivo}: {type(exc).__name__}: {exc}')

        if numero == 1 or numero % 250 == 0 or numero == len(archivos):
            print(f'Conteo completo: {numero:,}/{len(archivos):,} archivos.')

    filas = [
        {
            'codigoestacion': clave[0],
            'codigosensor': clave[1],
            **valores,
            'alcance': 'completo',
        }
        for clave, valores in acumulado.items()
    ]
    if not filas:
        return pd.DataFrame(columns=[
            'codigoestacion', 'codigosensor', 'registros',
            'fecha_min', 'fecha_max', 'alcance',
        ])
    return pd.DataFrame(filas).sort_values(
        'registros',
        ascending=False,
    ).reset_index(drop=True)


## 7. Hallazgos y severidad

Los hallazgos resumen síntomas observados. Una alerta basada en muestra no afirma que todo el dataset tenga el problema; indica qué debe confirmarse o resolverse antes del procesamiento diario.

Severidades:

- CRITICO: impide una transformación confiable.
- ADVERTENCIA: requiere una regla explícita o revisión.
- INFORMATIVO: describe el alcance o comportamiento.


In [26]:
def construir_hallazgos(
    inventario,
    resumen_particiones,
    presencia_columnas,
    muestra,
    conversiones,
    coordenadas_particion,
    unidades,
    resumen_duplicados,
    frecuencia,
    geografia,
    valores_sospechosos,
):
    hallazgos = []

    def agregar(severidad, categoria, metrica, valor, alcance, recomendacion):
        hallazgos.append({
            'severidad': severidad,
            'categoria': categoria,
            'metrica': metrica,
            'valor': str(valor),
            'alcance': alcance,
            'recomendacion': recomendacion,
        })

    agregar(
        'INFORMATIVO',
        'alcance',
        'archivos_auditados',
        len(inventario),
        'inventario_completo',
        'Conservar este valor como trazabilidad de la corrida.',
    )
    agregar(
        'INFORMATIVO',
        'alcance',
        'filas_inventariadas',
        int(inventario['filas'].fillna(0).sum()) if not inventario.empty else 0,
        'inventario_completo',
        'Conteo exacto obtenido desde metadatos Parquet.',
    )
    agregar(
        'INFORMATIVO',
        'alcance',
        'filas_muestra',
        len(muestra),
        'muestra_estratificada',
        'No interpretar conteos de la muestra como totales del dataset.',
    )

    errores_archivo = int(inventario['error_lectura'].notna().sum()) if not inventario.empty else 0
    if errores_archivo:
        agregar(
            'CRITICO',
            'archivos',
            'archivos_no_legibles',
            errores_archivo,
            'inventario_completo',
            'Reparar o volver a descargar los Parquet afectados.',
        )

    partes_con_huecos = 0
    if not resumen_particiones.empty:
        partes_con_huecos = int(
            resumen_particiones['partes_faltantes'].apply(bool).sum()
        )
    if partes_con_huecos:
        agregar(
            'CRITICO',
            'particiones',
            'particiones_con_huecos',
            partes_con_huecos,
            'inventario_completo',
            'No procesar hasta completar o justificar las partes faltantes.',
        )

    presentes = set(
        presencia_columnas.loc[
            presencia_columnas['archivos_presente'] > 0,
            'columna',
        ]
    ) if not presencia_columnas.empty else set()
    faltantes = sorted(set(COLUMNAS_ESPERADAS) - presentes)
    if faltantes:
        agregar(
            'CRITICO',
            'esquema',
            'columnas_esperadas_ausentes',
            ', '.join(faltantes),
            'inventario_completo',
            'Homologar el esquema antes de combinar variables o años.',
        )

    incompletas = []
    if not presencia_columnas.empty:
        incompletas = presencia_columnas.loc[
            presencia_columnas['esperada']
            & (presencia_columnas['archivos_presente'] < presencia_columnas['archivos_totales']),
            'columna',
        ].tolist()
    if incompletas:
        agregar(
            'CRITICO',
            'esquema',
            'columnas_ausentes_en_algunos_archivos',
            ', '.join(incompletas),
            'inventario_completo',
            'Homologar o volver a descargar los archivos con esquema incompleto.',
        )

    firmas = int(inventario['firma_esquema'].dropna().nunique()) if not inventario.empty else 0
    if firmas > 1:
        agregar(
            'ADVERTENCIA',
            'esquema',
            'variantes_de_esquema',
            firmas,
            'inventario_completo',
            'Revisar columnas y tipos por archivo antes de concatenar.',
        )

    fallos_conversion = int(conversiones['conversiones_fallidas'].sum()) if not conversiones.empty else 0
    if fallos_conversion:
        agregar(
            'CRITICO',
            'tipos',
            'conversiones_fallidas',
            fallos_conversion,
            'muestra_estratificada',
            'Definir reglas para fechas, valores o coordenadas no interpretables.',
        )

    problemas_coordenadas = 0
    if not coordenadas_particion.empty:
        problemas_coordenadas = int(coordenadas_particion['registros_muestra'].sum())
    if problemas_coordenadas:
        agregar(
            'CRITICO',
            'geografia',
            'coordenadas_o_particion_invalidas',
            problemas_coordenadas,
            'muestra_estratificada',
            'Corregir o excluir registros inválidos antes de asignar estaciones a municipios.',
        )

    if not resumen_duplicados.empty:
        exactos = int(resumen_duplicados.iloc[0]['filas_duplicadas_exactas'])
        claves = int(resumen_duplicados.iloc[0]['claves_duplicadas'])
        conflictos = int(resumen_duplicados.iloc[0]['claves_con_valores_conflictivos'])

        if exactos:
            agregar(
                'ADVERTENCIA',
                'duplicados',
                'filas_duplicadas_exactas',
                exactos,
                'muestra_estratificada',
                'Definir deduplicación exacta antes de agregar por día.',
            )
        if claves:
            agregar(
                'ADVERTENCIA',
                'duplicados',
                'claves_repetidas',
                claves,
                'muestra_estratificada',
                'Revisar estación, sensor y timestamp antes de escoger un registro.',
            )
        if conflictos:
            agregar(
                'CRITICO',
                'duplicados',
                'claves_con_valores_distintos',
                conflictos,
                'muestra_estratificada',
                'No promediar conflictos automáticamente; investigar su origen.',
            )

    if len(unidades) > 1:
        agregar(
            'ADVERTENCIA',
            'unidades',
            'unidades_distintas',
            len(unidades),
            'muestra_estratificada',
            'Separar o convertir unidades antes de cualquier agregación.',
        )

    if not frecuencia.empty:
        cadencias = int(frecuencia['intervalo_moda_segundos'].dropna().nunique())
        if cadencias > 1:
            agregar(
                'ADVERTENCIA',
                'frecuencia',
                'cadencias_modales_distintas',
                cadencias,
                'muestra_bloques_contiguos',
                'Definir cobertura diaria por sensor; no ponderar por número bruto de filas.',
            )

    if not geografia.empty:
        estaciones_inconsistentes = int((
            (geografia['departamentos'] > 1)
            | (geografia['municipios'] > 1)
            | (geografia['coordenadas_redondeadas'] > 1)
        ).sum())
        if estaciones_inconsistentes:
            agregar(
                'ADVERTENCIA',
                'geografia',
                'estaciones_con_geografia_variable',
                estaciones_inconsistentes,
                'muestra_estratificada',
                'Revisar traslados, etiquetas y coordenadas antes de asignar municipios.',
            )

    if not valores_sospechosos.empty:
        agregar(
            'ADVERTENCIA',
            'valores',
            'valores_fuera_rango_configurado',
            len(valores_sospechosos),
            'muestra_estratificada',
            'Validar unidad y sensor antes de excluir o marcar estos valores.',
        )

    if len(hallazgos) == 3:
        agregar(
            'INFORMATIVO',
            'resultado',
            'sin_alertas_en_muestra',
            True,
            'muestra_estratificada',
            'La ausencia de alertas en una muestra no demuestra ausencia de problemas.',
        )

    orden = {'CRITICO': 0, 'ADVERTENCIA': 1, 'INFORMATIVO': 2}
    resultado = pd.DataFrame(hallazgos)
    resultado['_orden'] = resultado['severidad'].map(orden)
    return resultado.sort_values(
        ['_orden', 'categoria', 'metrica']
    ).drop(columns='_orden').reset_index(drop=True)


def guardar_tablas(tablas):
    salida = (
        PROCESSED_ROOT
        / 'auditorias'
        / f'variable={slugificar(VARIABLE_NOMBRE)}'
        / f'fuente={str(DATASET_ID).lower()}'
        / f'ejecucion={slugificar(ETIQUETA_SALIDA)}'
    )
    salida.mkdir(parents=True, exist_ok=True)

    for nombre, tabla in tablas.items():
        if isinstance(tabla, pd.DataFrame) and not tabla.empty:
            tabla.to_parquet(salida / f'{nombre}.parquet', index=False)

    print(f'Resultados guardados en: {salida}')
    return salida


## 8. Ejecución

La ejecución muestra primero el inventario exacto y después los resultados de la muestra. Los DataFrames completos permanecen disponibles en memoria; en pantalla se presentan resúmenes y ejemplos controlados.


In [27]:
tablas_auditoria = {}
hallazgos = pd.DataFrame()

departamentos, anios, meses = validar_configuracion()
raiz = raiz_dataset()

print(f'Ruta auditada: {raiz}')
print(f'Auditoría activada: {EJECUTAR_AUDITORIA}')

if not EJECUTAR_AUDITORIA:
    print(
        'Auditoría desactivada. Revise la configuración y cambie '
        'EJECUTAR_AUDITORIA a True.'
    )
else:
    archivos = descubrir_archivos(
        raiz=raiz,
        departamentos=departamentos,
        anios=anios,
        meses=meses,
    )
    if not archivos:
        raise FileNotFoundError(
            'No se encontraron Parquet para los filtros configurados. '
            f'Ruta base: {raiz}'
        )

    display(Markdown('## Inventario exacto'))
    inventario_archivos = construir_inventario(archivos, raiz)
    resumen_particiones = resumir_particiones(inventario_archivos)
    presencia_columnas, tipos_columnas = resumir_esquemas(inventario_archivos)

    print(f'Archivos: {len(inventario_archivos):,}')
    print(f"Filas: {int(inventario_archivos['filas'].fillna(0).sum()):,}")
    print(
        'Tamaño: '
        f"{inventario_archivos['tamano_bytes'].fillna(0).sum() / (1024 ** 2):,.2f} MB"
    )
    display(resumen_particiones.head(60))
    display(presencia_columnas)
    display(tipos_columnas)

    display(Markdown('## Muestra estratificada'))
    seleccion_muestra = seleccionar_archivos_muestra(inventario_archivos)
    print(f'Archivos seleccionados: {len(seleccion_muestra):,}')
    print(f"Filas estimadas: {int(seleccion_muestra['filas'].fillna(0).sum()):,}")
    display(
        seleccion_muestra[
            ['departamento', 'anio', 'mes', 'parte', 'filas', 'bloque_muestra']
        ].head(60)
    )

    muestra_cruda = cargar_muestra(seleccion_muestra)
    muestra = preparar_muestra(muestra_cruda)
    print(f'Filas cargadas en muestra: {len(muestra):,}')

    nulos = auditar_nulos(muestra)
    conversiones = auditar_conversiones(muestra)
    coordenadas_particion = auditar_coordenadas_y_particion(muestra)
    unidades = auditar_unidades(muestra)
    resumen_valores, valores_sospechosos = auditar_valores(muestra)
    (
        resumen_duplicados,
        ejemplos_duplicados_exactos,
        claves_duplicadas,
        conflictos,
    ) = auditar_duplicados(muestra)

    (
        conteo_estaciones_muestra,
        frecuencia_estaciones,
        densidad_diaria,
    ) = resumir_estaciones_muestra(muestra)
    geografia_estaciones = auditar_geografia(muestra)

    conteo_estaciones = conteo_estaciones_muestra
    if EJECUTAR_CONTEO_ESTACIONES_COMPLETO:
        conteo_estaciones = contar_estaciones_completo(archivos)

    hallazgos = construir_hallazgos(
        inventario=inventario_archivos,
        resumen_particiones=resumen_particiones,
        presencia_columnas=presencia_columnas,
        muestra=muestra,
        conversiones=conversiones,
        coordenadas_particion=coordenadas_particion,
        unidades=unidades,
        resumen_duplicados=resumen_duplicados,
        frecuencia=frecuencia_estaciones,
        geografia=geografia_estaciones,
        valores_sospechosos=valores_sospechosos,
    )

    display(Markdown('## Calidad y valores'))
    display(nulos)
    display(conversiones)
    display(coordenadas_particion)
    display(unidades)
    display(resumen_valores)
    display(resumen_duplicados)

    display(Markdown('## Estaciones y frecuencia'))
    display(conteo_estaciones.head(30))
    display(frecuencia_estaciones.head(30))
    display(densidad_diaria.head(30))
    display(geografia_estaciones.head(30))

    display(Markdown('## Hallazgos'))
    display(hallazgos)

    tablas_auditoria = {
        'inventario_archivos': inventario_archivos.drop(
            columns=['columnas', 'esquema'],
            errors='ignore',
        ),
        'resumen_particiones': resumen_particiones,
        'presencia_columnas': presencia_columnas,
        'tipos_columnas': tipos_columnas,
        'seleccion_muestra': seleccion_muestra.drop(
            columns=['columnas', 'esquema'],
            errors='ignore',
        ),
        'nulos': nulos,
        'conversiones': conversiones,
        'coordenadas_particion': coordenadas_particion,
        'unidades': unidades,
        'resumen_valores': resumen_valores,
        'resumen_duplicados': resumen_duplicados,
        'ejemplos_duplicados_exactos': ejemplos_duplicados_exactos,
        'claves_duplicadas': claves_duplicadas,
        'conflictos': conflictos,
        'conteo_estaciones': conteo_estaciones,
        'frecuencia_estaciones': frecuencia_estaciones,
        'densidad_diaria': densidad_diaria,
        'geografia_estaciones': geografia_estaciones,
        'valores_sospechosos': valores_sospechosos,
        'hallazgos': hallazgos,
    }

    if GUARDAR_RESULTADOS:
        guardar_tablas(tablas_auditoria)


Ruta auditada: /content/drive/MyDrive/eco2026_processed/clima_crudo/variable=humedad/fuente=uext-mhny
Auditoría activada: True


## Inventario exacto

Archivos: 633
Filas: 627,529
Tamaño: 10.24 MB


,departamento,anio,mes,archivos,filas,tamano_mb,primera_parte,ultima_parte,partes_faltantes,errores_lectura
0,CUNDINAMARCA,2025,1,80,79454,1.05,0,79,[],0
1,CUNDINAMARCA,2025,2,12,11524,0.16,0,11,[],0
2,CUNDINAMARCA,2025,3,37,36815,0.56,0,36,[],0
3,CUNDINAMARCA,2025,4,37,36672,0.56,0,36,[],0
4,CUNDINAMARCA,2025,5,37,36006,0.55,0,36,[],0
5,CUNDINAMARCA,2025,6,48,47454,0.75,0,47,[],0
6,CUNDINAMARCA,2025,7,63,62359,1.08,0,62,[],0
7,CUNDINAMARCA,2025,8,48,47646,0.78,0,47,[],0
8,CUNDINAMARCA,2025,9,51,50681,0.86,0,50,[],0
9,CUNDINAMARCA,2025,10,64,63469,1.10,0,63,[],0


,columna,archivos_presente,archivos_totales,esperada
0,codigoestacion,633,633,True
1,codigosensor,633,633,True
2,dataset_id,633,633,True
3,departamento,633,633,True
4,descripcionsensor,633,633,True
5,fechaobservacion,633,633,True
6,latitud,633,633,True
7,longitud,633,633,True
8,municipio,633,633,True
9,nombreestacion,633,633,True


...

,codigoestacion,codigosensor,intervalos_muestra,intervalo_moda_segundos,intervalo_mediano_segundos,intervalo_p10_segundos,intervalo_p90_segundos,intervalo_min_segundos,intervalo_max_segundos,alcance
0,0021195120,0027,618,3600.0,3600.0,3600.0,3600.0,3600.0,3600.0,muestra_bloques_contiguos
1,0021195170,0027,435,3600.0,3600.0,3600.0,3600.0,3600.0,54000.0,muestra_bloques_contiguos
2,0021195190,0027,196,3600.0,3600.0,3600.0,3600.0,3600.0,3600.0,muestra_bloques_contiguos
3,0021205420,0027,585,3600.0,3600.0,3600.0,3600.0,3600.0,3600.0,muestra_bloques_contiguos
4,0021205511,0027,402,600.0,600.0,600.0,600.0,600.0,10800.0,muestra_bloques_contiguos
5,0021205513,0027,212,600.0,600.0,600.0,3600.0,600.0,3600.0,muestra_bloques_contiguos
6,0021205515,0027,967,600.0,600.0,600.0,600.0,600.0,3600.0,muestra_bloques_contiguos
7,0021205517,0027,425,600.0,600.0,600.0,600.0,600.0,3600.0,muestra_bloques_contiguos
8,0021205518,0027,385,600.0,600.0,600.0,3600.0,600.0,14400.0,muestra_bloques_contiguos
9,0021205519,0027,346,600.0,600.0,600.0,600.0,600.0,3600.0,muestra_bloques_contiguos


,codigoestacion,codigosensor,dias_en_muestra,observaciones_dia_min,observaciones_dia_mediana,observaciones_dia_max,alcance
0,0021195120,0027,50,1,15.5,36,muestra
1,0021195170,0027,40,1,11.0,36,muestra
2,0021195190,0027,18,1,12.0,24,muestra
3,0021205420,0027,47,1,16.0,36,muestra
4,0021205511,0027,7,25,43.0,107,muestra
5,0021205513,0027,7,1,19.0,107,muestra
6,0021205515,0027,14,11,87.5,133,muestra
7,0021205517,0027,6,11,87.5,109,muestra
8,0021205518,0027,20,1,16.5,101,muestra
9,0021205519,0027,6,1,63.5,109,muestra


,codigoestacion,nombres_estacion,departamentos,municipios,coordenadas_redondeadas,latitud_min,latitud_max,longitud_min,longitud_max
0,0021195120,1,1,1,1,4.392920,4.392920,-74.392720,-74.392720
1,0021195170,1,1,1,1,3.993611,3.993611,-74.398056,-74.398056
2,0021195190,1,1,1,1,4.307189,4.307189,-74.308399,-74.308399
3,0021205420,1,1,1,2,4.688662,4.691417,-74.209000,-74.205626
4,0021205511,1,1,1,1,5.001111,5.001111,-73.733333,-73.733333
5,0021205513,1,1,1,1,4.504577,4.504577,-74.184187,-74.184187
6,0021205515,1,1,1,1,4.700289,4.700289,-74.173786,-74.173786
7,0021205517,1,1,1,1,4.788157,4.788157,-74.050059,-74.050059
8,0021205518,1,1,1,1,5.171361,5.171361,-74.015667,-74.015667
9,0021205519,1,1,1,1,4.452111,4.452111,-74.272288,-74.272288


## Hallazgos

,severidad,categoria,metrica,valor,alcance,recomendacion
0,ADVERTENCIA,duplicados,claves_repetidas,3000,muestra_estratificada,"Revisar estación, sensor y timestamp antes de ..."
1,ADVERTENCIA,duplicados,filas_duplicadas_exactas,6000,muestra_estratificada,Definir deduplicación exacta antes de agregar ...
2,ADVERTENCIA,esquema,variantes_de_esquema,2,inventario_completo,Revisar columnas y tipos por archivo antes de ...
3,ADVERTENCIA,frecuencia,cadencias_modales_distintas,3,muestra_bloques_contiguos,Definir cobertura diaria por sensor; no ponder...
4,ADVERTENCIA,geografia,estaciones_con_geografia_variable,13,muestra_estratificada,"Revisar traslados, etiquetas y coordenadas ant..."
5,INFORMATIVO,alcance,archivos_auditados,633,inventario_completo,Conservar este valor como trazabilidad de la c...
6,INFORMATIVO,alcance,filas_inventariadas,627529,inventario_completo,Conteo exacto obtenido desde metadatos Parquet.
7,INFORMATIVO,alcance,filas_muestra,48000,muestra_estratificada,No interpretar conteos de la muestra como tota...


## 9. Interpretación y siguiente paso

Este notebook responde qué síntomas presenta la fuente y con qué alcance fueron observados. No decide automáticamente qué fila conservar ni cómo resumir una variable.

Antes de desarrollar 03_ClimateDailyProcessor deben quedar explícitas, por variable:

- La semántica de valorobservado y unidadmedida.
- La clave definitiva de deduplicación.
- El tratamiento de conflictos.
- Los rangos físicamente plausibles.
- La frecuencia o frecuencias esperadas por sensor.
- La cobertura mínima para aceptar un día.
- Las estadísticas diarias apropiadas.
- Las banderas de calidad que acompañarán cada agregado.

Los huecos del calendario y una posible imputación se evalúan después de crear la capa diaria. No se imputan observaciones subdiarias por defecto.
